# LangChain Fundamentals — A Hands-On Notebook
---
**What this is:** A self-contained notebook that walks you through how AI applications are actually built — from raw prompts to intelligent agents.

**What you'll learn:**
- How LLMs work under the hood (they predict the next word, they don't "think")
- The 4-step pipeline: Prompt Template → LLM → Output Parser → Chains/Agents
- 6 prompting techniques — from basic to advanced
- How to build chains, memory, and agents with LangChain

**Requirements:** Python 3.9+ and an OpenAI API key (even the free tier works — we use `gpt-4o-mini`).

> Built by [Saif Ali Khan](https://www.linkedin.com/in/YOUR-PROFILE).

---
## 0. Setup — Install & Configure
Run the cell below to install the required packages. You only need to do this once.

In [ ]:
%%capture
!pip install langchain langchain-openai langchain-experimental pandas tabulate

### Add your OpenAI API key
Replace `"your-api-key-here"` with your actual key. You can get one at [platform.openai.com/api-keys](https://platform.openai.com/api-keys).

> **Tip:** Never share your API key publicly. If you push this to GitHub, use environment variables instead.

In [ ]:
import os

# Option 1: Paste your key directly (for quick testing)
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# Option 2 (recommended): Set it as an environment variable before launching Jupyter
# Then you can just do: os.environ.get("OPENAI_API_KEY")

---
## 1. The LLM — Your AI Brain

An LLM (Large Language Model) doesn't understand language. It **predicts the next word** — thousands of times in a row — to produce a response. What makes it useful are the **parameters** that control its behavior:

| Parameter | What It Does | Analogy |
|-----------|-------------|---------|
| `temperature` | Controls randomness. 0 = always pick the safest word. 1 = creative and unpredictable. | Thermostat for creativity |
| `top_p` | Only consider words that make up the top P% of probability. Adaptive. | Adaptive word filter |
| `top_k` | Only consider the top K most likely words. Fixed. | Fixed headcount |
| `max_tokens` | Maximum length of the response. | Word limit on an essay |

When you use ChatGPT, these parameters are **pre-tuned for you**. Here, you control them yourself.

In [ ]:
from langchain_openai import ChatOpenAI

# Create an LLM with specific parameters
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,     # balanced creativity
    max_tokens=256        # keep responses concise
)

# A simple helper function — pass in any prompt, get a response
def ask(prompt_text):
    response = llm.invoke(prompt_text)
    return response.content

# Test it!
print(ask("What is LangChain in one sentence?"))

---
## 2. Prompting Techniques — How You Ask Matters

The same model gives wildly different answers depending on **how you phrase the prompt**. Here are 6 techniques, from basic to advanced.

### 2a. Basic Prompt
No instructions. No examples. Just a sentence for the model to complete.

In [ ]:
# BASIC — just complete the sentence
print(ask("The capital of France is"))

### 2b. Zero-Shot
Give instructions, but no examples. The model figures it out from the task description alone.

In [ ]:
# ZERO-SHOT — instructions only, no examples
print(ask("Classify the following sentence as positive, negative, or neutral: 'The movie was absolutely brilliant and I loved every minute of it.'"))

### 2c. One-Shot
Provide **one example** so the model understands the pattern you want.

In [ ]:
# ONE-SHOT — one example, then the real task
prompt = """Classify the sentiment:

Example:
Text: "The food was terrible." → Sentiment: Negative

Now classify:
Text: "I had an amazing time at the concert!" → Sentiment:"""

print(ask(prompt))

### 2d. Few-Shot
Multiple examples (2–5). The model picks up the pattern much more reliably.

In [ ]:
# FEW-SHOT — multiple examples establish a clear pattern
prompt = """Convert the following company descriptions to short taglines.

Company: A ride-sharing app that connects drivers with passengers.
Tagline: Your ride, one tap away.

Company: An online marketplace for handmade crafts and vintage items.
Tagline: Handmade finds, delivered to your door.

Company: A cloud-based project management tool for remote teams.
Tagline: Teamwork, without the office.

Company: An AI-powered personal finance app that tracks spending and investments.
Tagline:"""

print(ask(prompt))

### 2e. Chain-of-Thought (CoT)
Force the model to **show its reasoning step by step**. Critical for math and logic tasks.

In [ ]:
# CHAIN-OF-THOUGHT — force the model to think step by step
prompt = """Solve this step by step:

A store sells apples for $2 each. A customer buys 5 apples and pays with a $20 bill. 
The store also applies a 10% discount for buying more than 3 apples.
How much change does the customer receive?

Let's think step by step:"""

print(ask(prompt))

### 2f. Self-Consistency
Run the **same question 3 times independently**, then pick the answer that appears most often. Reduces randomness.

In [ ]:
# SELF-CONSISTENCY — 3 independent answers, pick the majority
from collections import Counter

question = "Is a tomato a fruit or a vegetable? Think about this from a botanical and culinary perspective and give a one-word answer: fruit or vegetable."

# Use higher temperature for variety
creative_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9, max_tokens=50)

answers = []
for i in range(3):
    response = creative_llm.invoke(question)
    answer = response.content.strip().lower()
    answers.append(answer)
    print(f"  Attempt {i+1}: {answer}")

print(f"\nAll answers: {answers}")
print(f"Most common: {Counter(answers).most_common(1)[0][0]}")

---
## 3. The Pipeline — Prompt Template → LLM → Output Parser

This is the heart of LangChain. Instead of writing raw prompts every time, you build a **reusable pipeline** using three components connected by the **pipe operator `|`** (called LCEL — LangChain Expression Language).

### 3a. PromptTemplate — Reusable prompts with {placeholders}

In [ ]:
from langchain_core.prompts import PromptTemplate

# Create a template with two variables
template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {topic}."
)

# Fill in the blanks
filled = template.format(adjective="clever", topic="programming")
print(filled)
# Output: "Tell me a clever joke about programming."


### 3b. StrOutputParser — Clean the response

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Without parser: you get an AIMessage object with metadata
raw = llm.invoke("Say hello")
print(f"Raw type:    {type(raw)}")
print(f"Raw content: {raw}")

print("---")

# With parser: you get a clean string
parser = StrOutputParser()
clean = parser.invoke(raw)
print(f"Clean type:    {type(clean)}")
print(f"Clean content: {clean}")

### 3c. The Full Chain — Connect everything with `|`
This is LCEL (LangChain Expression Language). The pipe `|` operator passes the output of one step as the input to the next.

In [ ]:
# THE FULL PIPELINE: template → LLM → parser
chain = template | llm | StrOutputParser()

# Run it — just pass in the variables
result = chain.invoke({"adjective": "witty", "topic": "data science"})
print(result)

### 3d. RunnableLambda — Plug any Python function into the chain

In [ ]:
from langchain_core.runnables import RunnableLambda

# A plain Python function
def shout(text):
    return text.upper() + "!!!"

# Wrap it so it works with the | pipe
shout_step = RunnableLambda(shout)

# Add it to the chain — response gets shouted
loud_chain = template | llm | StrOutputParser() | shout_step
result = loud_chain.invoke({"adjective": "short", "topic": "cats"})
print(result)

---
## 4. Chains — Connecting Multiple Pipelines

### Sequential Chain
Output of Step 1 feeds into Step 2. Like an assembly line.

In [ ]:
# SEQUENTIAL CHAIN — Step 1 generates a topic, Step 2 writes a poem about it

step1 = PromptTemplate.from_template(
    "Name one famous landmark in {country}. Just the name, nothing else."
) | llm | StrOutputParser()

step2 = PromptTemplate.from_template(
    "Write a 4-line poem about {landmark}."
) | llm | StrOutputParser()

# Chain them: step1 output → step2 input
landmark = step1.invoke({"country": "Japan"})
print(f"Landmark: {landmark}")
print(f"\nPoem:\n{step2.invoke({'landmark': landmark})}")

### RunnableParallel — Run multiple chains at the same time
Use a **dictionary** `{ }` to run tasks simultaneously.

In [ ]:
from langchain_core.runnables import RunnableParallel

# Three independent tasks running in parallel
parallel = RunnableParallel(
    joke=PromptTemplate.from_template("Tell a one-liner joke about {topic}") | llm | StrOutputParser(),
    fact=PromptTemplate.from_template("Tell me one fun fact about {topic}") | llm | StrOutputParser(),
    haiku=PromptTemplate.from_template("Write a haiku about {topic}") | llm | StrOutputParser(),
)

results = parallel.invoke({"topic": "coffee"})

print("JOKE:", results["joke"])
print("\nFACT:", results["fact"])
print("\nHAIKU:", results["haiku"])

---
## 5. Chat Messages & Memory

Chat models understand **roles** — who said what. This is how multi-turn conversations work.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# A conversation with roles
messages = [
    SystemMessage(content="You are a pirate. Always respond in pirate speak."),
    HumanMessage(content="What's the weather like today?"),
]

response = llm.invoke(messages)
print(response.content)

### Memory — Remember past conversations
Without memory, each call to the LLM starts fresh. `ChatMessageHistory` stores the conversation so context carries forward.

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory

# Create a memory store
memory = ChatMessageHistory()

# Simulate a conversation
memory.add_user_message("My name is Saif and I love building AI tools.")
memory.add_ai_message("Nice to meet you, Saif! What kind of AI tools are you building?")
memory.add_user_message("I'm learning LangChain to build intelligent agents.")

# The model now has full context
response = llm.invoke(memory.messages)
print(response.content)

# Check what's stored
print("\n--- Memory Contents ---")
for msg in memory.messages:
    role = type(msg).__name__.replace("Message", "")
    print(f"  {role}: {msg.content[:80]}...")

---
## 6. Agents — Let the LLM Decide

Chains follow a fixed path. Agents are different — you give the LLM a **toolbox** and it **decides what to do on its own**.

No output parser needed. The agent handles everything internally.

> **Analogy:** A chain is like written driving directions (same route every time). An agent is like a GPS — give it a destination, it figures out the best route.

In [ ]:
import pandas as pd
from langchain_experimental.agents import create_pandas_dataframe_agent

# Create a sample dataset
data = {
    "Product": ["Laptop", "Phone", "Tablet", "Headphones", "Monitor"],
    "Category": ["Electronics", "Electronics", "Electronics", "Accessories", "Electronics"],
    "Price": [999, 699, 449, 149, 349],
    "Units_Sold": [150, 300, 200, 500, 120],
    "Rating": [4.5, 4.7, 4.3, 4.6, 4.2]
}
df = pd.DataFrame(data)
print(df.to_string(index=False))

In [ ]:
# CREATE THE AGENT — give it the LLM brain and the dataframe as a tool
agent = create_pandas_dataframe_agent(
    llm, df, 
    verbose=True,              # show the agent's reasoning
    allow_dangerous_code=True,  # let it write Python code
    handle_parsing_errors=True
)

# Ask a simple question
print(agent.invoke("What is the most expensive product?")["output"])

In [ ]:
# Ask something complex — the agent writes its own code to answer
print(agent.invoke("What is the total revenue (price × units sold) for each product? Sort by revenue descending.")["output"])

In [ ]:
# Even more complex — multi-step reasoning
print(agent.invoke("Which category has the highest average rating, and what's the price range in that category?")["output"])

---
## What You Just Learned

You've built every component of the AI pipeline that powers tools like ChatGPT:

```
User Input → Prompt Template → LLM → Output Parser → Final Output
               |                  |                |
          {placeholders}    temperature        StrOutputParser
          System Message     top_p              RunnableLambda
          Chat History       max_tokens
```

**The key insight:** When you use ChatGPT, all of this is pre-tuned behind the scenes. When you build with LangChain + OpenAI, you control every single knob.

---

### LCEL Quick Reference
| Syntax | Meaning | Example |
|--------|---------|---------|
| `\|` (pipe) | Sequential — run one after another | `prompt \| llm \| parser` |
| `{ }` (dict) | Parallel — run all at the same time | `RunnableParallel(a=..., b=...)` |
| `RunnableLambda` | Wrap any Python function | `RunnableLambda(my_func)` |
| `StrOutputParser` | Strip metadata, return clean text | `chain = prompt \| llm \| StrOutputParser()` |

---

*Built by [Saif Ali Khan](https://www.linkedin.com/in/YOUR-PROFILE) · Powered by OpenAI GPT-4o-mini · 2026*
